# Colab BGE-M3 Reranker API

Colab GPU에는 BGE-M3만 적재하고 로컬 Customer Intelligence가 HTTPS로 `/rerank`를 호출합니다. Colab의 Secrets에 `NGROK_AUTHTOKEN`과 충분히 긴 임의 값인 `RERANKER_API_KEY`를 등록한 뒤 GPU 런타임에서 전체 셀을 실행하세요. 런타임이 종료되면 URL도 만료됩니다.

In [4]:
%pip install -q FlagEmbedding==1.4.2 fastapi uvicorn pyngrok

In [5]:
from getpass import getpass
import torch

if not torch.cuda.is_available():
    raise RuntimeError("GPU 런타임을 선택하세요.")

NGROK_AUTHTOKEN = getpass("NGROK_AUTHTOKEN 입력: ").strip()
RERANKER_API_KEY = getpass("RERANKER_API_KEY 입력: ").strip()

if not NGROK_AUTHTOKEN or not RERANKER_API_KEY:
    raise RuntimeError("두 인증값을 모두 입력해야 합니다.")

In [6]:
from FlagEmbedding import BGEM3FlagModel

model = BGEM3FlagModel('BAAI/bge-m3', devices='cuda', use_fp16=True)
print('loaded: BAAI/bge-m3')

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

...

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

loaded: BAAI/bge-m3


In [7]:
import secrets
import threading

import uvicorn
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel, Field
from pyngrok import ngrok

class RerankRequest(BaseModel):
    query: str = Field(min_length=1)
    passages: list[str] = Field(min_length=1, max_length=20)

app = FastAPI()

@app.get('/health')
def health():
    return {'status': 'ok', 'model': 'BAAI/bge-m3'}

@app.post('/rerank')
def rerank(payload: RerankRequest, authorization: str | None = Header(default=None)):
    expected = f'Bearer {RERANKER_API_KEY}'
    if authorization is None or not secrets.compare_digest(authorization, expected):
        raise HTTPException(status_code=401, detail='invalid reranker token')
    pairs = [(payload.query, passage) for passage in payload.passages]
    output = model.compute_score(
        pairs,
        batch_size=2,
        max_query_length=256,
        max_passage_length=2048,
        weights_for_different_modes=[0.0, 0.0, 1.0],
    )['colbert']
    if isinstance(output, (int, float)):
        output = [output]
    return {'scores': [float(score) for score in output]}

ngrok.set_auth_token(NGROK_AUTHTOKEN)
ngrok.kill()
server = uvicorn.Server(uvicorn.Config(app, host='0.0.0.0', port=8003, log_level='warning'))
threading.Thread(target=server.run, daemon=True).start()
public_url = ngrok.connect(8003, 'http').public_url
print('RAG_RERANKER_BASE_URL=' + public_url)

RAG_RERANKER_BASE_URL=https://ferment-pedometer-onstage.ngrok-free.dev                              
